In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Complete_DataFrame_Transformations_Actions")
    .master("local[*]")   
    .getOrCreate()
)

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 22:15:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/23 22:15:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
employee_data = [
    (101, "Anuj",   "IT",      90000, 29, "Mumbai",    1001, "2023-01-10", 4.7, None),
    (102, "Riya",   "HR",      60000, 31, "Pune",      1002, "2022-11-05", 4.2, "A"),
    (103, "Vikas",  "IT",      85000, 27, "Bengaluru", 1001, "2024-02-12", 4.5, "B"),
    (104, "Sneha",  "Finance", 95000, 35, "Mumbai",    1003, "2021-08-19", 4.8, "A"),
    (105, "Amit",   "Sales",   50000, 26, "Delhi",     1004, "2023-06-01", 3.9, None),
    (106, "Pooja",  "HR",      65000, 30, "Chennai",   1002, "2020-12-11", 4.1, "B"),
    (107, "Karan",  "IT",     120000, 33, "Hyderabad", 1001, "2019-03-14", 4.9, "A"),
    (108, "Meera",  "Sales",   52000, 28, "Pune",      1004, "2024-01-18", 3.7, "C"),
    (109, "Rohit",  "Finance", 88000, 32, "Delhi",     1003, "2022-09-09", 4.0, "B"),
    (110, "Nisha",  "Ops",     70000, 29, "Mumbai",    1005, "2023-04-22", 4.3, None),
]

employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("emp_name", StringType(), False),
    StructField("dept", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("manager_id", IntegerType(), True),
    StructField("join_date", StringType(), True),
    StructField("rating", DoubleType(), True),
    StructField("grade", StringType(), True),
])

emp_df = spark.createDataFrame(employee_data, employee_schema) \
    .withColumn("join_date", F.to_date("join_date"))

In [3]:
emp_df.show()

+------+--------+-------+------+---+---------+----------+----------+------+-----+
|emp_id|emp_name|   dept|salary|age|     city|manager_id| join_date|rating|grade|
+------+--------+-------+------+---+---------+----------+----------+------+-----+
|   101|    Anuj|     IT| 90000| 29|   Mumbai|      1001|2023-01-10|   4.7| NULL|
|   102|    Riya|     HR| 60000| 31|     Pune|      1002|2022-11-05|   4.2|    A|
|   103|   Vikas|     IT| 85000| 27|Bengaluru|      1001|2024-02-12|   4.5|    B|
|   104|   Sneha|Finance| 95000| 35|   Mumbai|      1003|2021-08-19|   4.8|    A|
|   105|    Amit|  Sales| 50000| 26|    Delhi|      1004|2023-06-01|   3.9| NULL|
|   106|   Pooja|     HR| 65000| 30|  Chennai|      1002|2020-12-11|   4.1|    B|
|   107|   Karan|     IT|120000| 33|Hyderabad|      1001|2019-03-14|   4.9|    A|
|   108|   Meera|  Sales| 52000| 28|     Pune|      1004|2024-01-18|   3.7|    C|
|   109|   Rohit|Finance| 88000| 32|    Delhi|      1003|2022-09-09|   4.0|    B|
|   110|   Nisha

# Count total emp 
# Count emp per dept 
# Total salary per dept 
# Avg sal per dept 
# max sal per dept 


# Count the employees per (dept,city)
# Avg rating per dept 
# Avg sal of emp with rating > 4.5 
# Dept with highest avg sal


# Diff between max and min sala per department 

In [4]:
import pyspark.sql.functions as F

In [5]:
emp_df.count()

10

In [8]:
emp_df.groupBy("dept").count().show()

+-------+-----+
|   dept|count|
+-------+-----+
|     IT|    3|
|     HR|    2|
|  Sales|    2|
|Finance|    2|
|    Ops|    1|
+-------+-----+



In [9]:
emp_df.groupBy("dept").sum("salary").show()

+-------+-----------+
|   dept|sum(salary)|
+-------+-----------+
|     IT|     295000|
|     HR|     125000|
|  Sales|     102000|
|Finance|     183000|
|    Ops|      70000|
+-------+-----------+



In [11]:
emp_df.groupBy("dept").agg(
    F.sum("salary").alias("total Salary"),
    F.avg("salary").alias("Avg Salary"),
    F.max("salary").alias("max Salary"),
).show()

+-------+------------+-----------------+----------+
|   dept|total Salary|       Avg Salary|max Salary|
+-------+------------+-----------------+----------+
|     IT|      295000|98333.33333333333|    120000|
|     HR|      125000|          62500.0|     65000|
|  Sales|      102000|          51000.0|     52000|
|Finance|      183000|          91500.0|     95000|
|    Ops|       70000|          70000.0|     70000|
+-------+------------+-----------------+----------+



# Depatment where average salary > 80k 



In [ ]:
emp_df.groupBy("dept").agg(
    F.avg("salary").alias("avg_Salary")
).filter(F.col("avg_salary")> 80000).show()

+-------+-----------------+
|   dept|       avg_Salary|
+-------+-----------------+
|     IT|98333.33333333333|
|Finance|          91500.0|
+-------+-----------------+



26/03/24 00:46:40 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 914525 ms exceeds timeout 120000 ms
26/03/24 00:46:40 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/24 00:46:40 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$